# DiD-BCF — B2_sweep_serial (N=800, linearity_degree=3)

**Workstream B2 · canonical DiD with AR(1) serially-correlated errors

sample-size sweep under serial correlation**

Fits DiD-BCF on this scenario at the **single sample size N above** and reports
metrics for both the plain DiD-BCF posterior and the proposed posterior
correction. The B2 sample-size sweep is split one-N-per-notebook to keep run
time manageable (combine the four N CSVs afterwards for the sqrt(N) analysis).

> **Colab:** upload just this notebook and *Run all*.


In [ ]:
# Colab: install the DiD-BCF dependencies (stochtree provides the BCF sampler).
%pip install -q stochtree scikit-learn joblib tqdm pandas numpy

In [ ]:
import os, sys

# --- Locate the DiD-BCF engine ------------------------------------------------
# So you can upload just THIS notebook to Colab and Run all. Resolution order:
#   1. `did_bcf_revision` already importable;
#   2. running inside a repo checkout (the parent folder holds the package);
#   3. otherwise clone https://github.com/hugogobato/DiD-BCF and use it.
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
ENGINE_SUBDIR = os.path.join("DiD-BCF", "Simulation_Studies_Revision")

def _locate_root():
    try:
        import did_bcf_revision  # noqa: F401
        return os.path.dirname(os.path.dirname(did_bcf_revision.__file__))
    except Exception:
        pass
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.isdir(os.path.join(parent, "did_bcf_revision")):
        return parent
    if not os.path.isdir("DiD-BCF"):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return os.path.abspath(ENGINE_SUBDIR)

ROOT = _locate_root()
sys.path.insert(0, ROOT)
print("Using DiD-BCF engine at:", ROOT)

from did_bcf_revision.runner import run_named
from did_bcf_revision.metrics import (compute_metrics, plain_vs_corrected,
                                      surface_metrics)

In [ ]:
REPS = 100      # replications for this (N, linearity_degree)
JOBS = 1        # parallel reps (keep 1 on a single-core/GPU Colab)
N   = 800      # <-- this notebook's sample size (one of 200 / 400 / 800 / 1600)
LIN = 3        # linearity degree for this notebook

bcf_params = dict(num_gfr=50, num_mcmc=500, keep_every=5, num_chains=3)

# The B2 sweep is split across notebooks: this one runs ONLY N above. (The full
# sweep in one notebook is 4x the fits and N=1600 BCF MCMC is the slow part.)
from did_bcf_revision import config as cfg
from did_bcf_revision.runner import run_experiment
exp = cfg.get_experiment("B2_sweep_serial")
exp.reps = REPS
exp.n_values = (N,)
summaries = run_experiment(exp, linearity_degree=LIN, jobs=JOBS, bcf_params=bcf_params,
                           prop_method="logit", n_splits=2, save=False)
out_csv = f"summaries_B2_sweep_serial_N{N}_lin_{LIN}.csv"
summaries.to_csv(out_csv, index=False)
print("wrote", out_csv, "| rows:", len(summaries))
summaries.head()


In [ ]:
# Decomposed metrics: bias, MC SD/variance, RMSE, MAE, MAPE, coverage 90/95,
# interval length, calibration ratio (avg_post_sd/emp_sd), size/power and their
# Monte-Carlo SEs -- for plain AND corrected DiD-BCF.
metrics = compute_metrics(summaries)
plain_vs_corrected(metrics)

## CATT-surface metrics (the paper's headline RMSE/MAE/MAPE)

Within-replication RMSE/MAE/MAPE over the *individual* treated observations
(mean +/- SD across runs) plus the *pointwise* CATT coverage -- the evidence
that DiD-BCF recovers the heterogeneous effect that GATT-only methods cannot.

In [ ]:
surface_metrics(summaries)